# Bước 1: Tiền xử lý dữ liệu
## Đề 13: Phân tích hội thoại trong DVKH trực tuyến
**Mục tiêu:** Làm sạch, chuẩn hóa từ lóng, tách từ tiếng Việt, trích xuất TF-IDF

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/dataset_final.csv')
print(f'Loaded: {len(df)} dòng, {df["conv_id"].nunique()} đoạn')
df.head()

## 1.1 Xây dựng từ điển từ lóng

In [ ]:
slang_dict = pd.read_csv('../data/processed/slang_dict.csv')
print(f'Từ điển: {len(slang_dict)} từ lóng')
slang_dict.head(10)

In [ ]:
# Chuyển thành dictionary để lookup nhanh
slang_map = dict(zip(slang_dict['slang'], slang_dict['standard']))
print('Ví dụ:')
for k, v in list(slang_map.items())[:10]:
    print(f'  {k:10s} → {v}')

## 1.2 Hàm tiền xử lý

In [ ]:
def clean_text(text):
    """Làm sạch văn bản thô"""
    if pd.isna(text):
        return ''
    text = str(text)
    # Chuyển lowercase
    text = text.lower()
    # Loại URL
    text = re.sub(r'https?://\S+', '', text)
    # Loại email
    text = re.sub(r'\S+@\S+', '', text)
    # Giữ lại chữ cái tiếng Việt, số, khoảng trắng
    text = re.sub(r'[^a-zA-Zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ0-9\s]', ' ', text)
    # Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_slang(text, slang_map):
    """Chuẩn hóa từ lóng dựa trên từ điển"""
    words = text.split()
    normalized = []
    for word in words:
        normalized.append(slang_map.get(word, word))
    return ' '.join(normalized)


def preprocess_pipeline(text, slang_map):
    """Pipeline tiền xử lý hoàn chỉnh"""
    text = clean_text(text)
    text = normalize_slang(text, slang_map)
    return text


# Test
test_cases = [
    'shop ơi đơn e đặt 3 ngày r mà k thấy ship, sao v ạ',
    'sp bị lỗi mà shop k chịu đổi, tôi muốn hoàn tiền',
    'e mua hàng ck r mà chưa thấy giao, bh ms dc v???',
    'Dạ em xin lỗi vì sự bất tiện này ạ.',
]

print('TEST TIỀN XỬ LÝ:')
print('=' * 80)
for tc in test_cases:
    result = preprocess_pipeline(tc, slang_map)
    print(f'Trước: {tc}')
    print(f'Sau  : {result}')
    print('-' * 80)

## 1.3 Áp dụng tiền xử lý lên toàn bộ dataset

In [ ]:
# Áp dụng pipeline
df['message_clean'] = df['message'].apply(lambda x: preprocess_pipeline(x, slang_map))

# Loại bỏ dòng trống sau xử lý
empty_count = (df['message_clean'] == '').sum()
print(f'Số dòng trống sau xử lý: {empty_count}')
df = df[df['message_clean'] != ''].reset_index(drop=True)

print(f'Dataset sau tiền xử lý: {len(df)} dòng')
print()

# So sánh trước/sau
print('SO SÁNH TRƯỚC / SAU:')
print('=' * 100)
sample = df[df['speaker'] == 'customer'].sample(10, random_state=42)
for _, row in sample.iterrows():
    print(f'Trước: {row["message"]}')
    print(f'Sau  : {row["message_clean"]}')
    print('-' * 100)

## 1.4 Trích xuất đặc trưng TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Chỉ lấy message của customer để phân tích
customer_df = df[df['speaker'] == 'customer'].copy()
print(f'Số lượt lời khách hàng: {len(customer_df)}')

# TF-IDF
tfidf = TfidfVectorizer(
    max_features=3000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)
tfidf_matrix = tfidf.fit_transform(customer_df['message_clean'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Số features (từ vựng): {len(tfidf.get_feature_names_out())}')
print(f'Top 20 từ vựng: {list(tfidf.get_feature_names_out()[:20])}')

## 1.5 Lưu kết quả

In [ ]:
from scipy import sparse

# Lưu dataset đã xử lý
df.to_csv('../data/processed/cleaned_data.csv', index=False, encoding='utf-8-sig')
print('✅ Saved: cleaned_data.csv')

# Lưu TF-IDF matrix
sparse.save_npz('../data/processed/tfidf_matrix.npz', tfidf_matrix)
print('✅ Saved: tfidf_matrix.npz')

# Lưu vocabulary
vocab_df = pd.DataFrame({'word': tfidf.get_feature_names_out()})
vocab_df.to_csv('../data/processed/tfidf_vocab.csv', index=False)
print('✅ Saved: tfidf_vocab.csv')

# Lưu customer_df index để mapping
customer_df.to_csv('../data/processed/customer_messages.csv', index=False, encoding='utf-8-sig')
print('✅ Saved: customer_messages.csv')

print(f'\nTổng kết: {len(df)} dòng đã xử lý, {tfidf_matrix.shape[1]} features TF-IDF')